In [1]:
!pip install malaya nltk PySastrawi

In [2]:
import pandas as pd
import re
import malaya

C:\Users\user\anaconda3\envs\python310\lib\site-packages\malaya\tokenizer.py:214: FutureWarning: Possible nested set at position 3397
  self.tok = re.compile(r'({})'.format('|'.join(pipeline)))
C:\Users\user\anaconda3\envs\python310\lib\site-packages\malaya\tokenizer.py:214: FutureWarning: Possible nested set at position 3927
  self.tok = re.compile(r'({})'.format('|'.join(pipeline)))


In [3]:
# === Load Data ===
cols = ['Category', 'Question (Malay)', 'Answer (Malay)', 'Source URL', 'Question (English)', 'Answer (English)']
df = pd.read_csv('../Data/Dengue_QA_multilingual.csv', usecols=cols)
print("\n=== Preview: Raw Data ===")
print(df.head())
print(df.columns)


=== Preview: Raw Data ===
   Category            Question (Malay)  \
0         1              Apakah denggi?   
1         1             Apa itu denggi?   
2         1        Apakah demam denggi?   
3         1        Berapa jenis denggi?   
4         1  Apakah punca demam denggi?   

                                      Answer (Malay)  \
0  Ia adalah penyakit berpotensi maut yang terseb...   
1  Demam denggi adalah penyakit yang disebarkan o...   
2  Demam denggi adalah sejenis penyakit jangkitan...   
3  Terdapat empat jenis virus yang boleh menyebab...   
4  Demam denggi adalah jangkitan virus yang diseb...   

                                          Source URL  \
0  https://hellodoktor.com/berjangkit/denggi/6-ca...   
1  https://www.doctoroncall.com.my/health-centre/...   
2      https://idengue.mysa.gov.my/tanda_denggi.html   
3      https://idengue.mysa.gov.my/tanda_denggi.html   
4  https://www.pantai.com.my/ms/health-pulse/deng...   

                   Question (English)  \

In [4]:
# === Remove Duplicates ===
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f'\nDropped {before - after} duplicate rows.')


Dropped 0 duplicate rows.


In [5]:
# === Handle Missing Data ===
missing = df[df['Question (Malay)'].isnull() | df['Answer (Malay)'].isnull() | df['Question (English)'].isnull() | df['Answer (English)'].isnull()]
print("\n=== Rows with Missing Q or A ===")
print(missing)
df = df.dropna(subset=['Question (Malay)', 'Answer (Malay)', 'Question (English)', 'Answer (English)'])
print(f"\nAfter dropna, shape is {df.shape}")


=== Rows with Missing Q or A ===
Empty DataFrame
Columns: [Category, Question (Malay), Answer (Malay), Source URL, Question (English), Answer (English)]
Index: []

After dropna, shape is (149, 6)


In [6]:
# === Text Cleaning ===
def clean_question(text):
    text = str(text)
    text = re.sub(r'<.*?>', '', text)                       # Remove HTML tags
    text = re.sub(r'[^a-zA-Z0-9\u00C0-\u017F\s]', '', text) # Remove special chars
    text = re.sub(r'\s+', ' ', text)                        # Remove extra spaces
    return text.strip()

df['Question (Malay)'] = df['Question (Malay)'].apply(clean_question)
df['Question (English)'] = df['Question (English)'].apply(clean_question)
print("\n=== Preview: Cleaned Text ===")
print(df[['Question (Malay)', 'Question (English)']].head())


=== Preview: Cleaned Text ===
            Question (Malay)                 Question (English)
0              Apakah denggi                     What is dengue
1             Apa itu denggi                     What is dengue
2        Apakah demam denggi               What is dengue fever
3        Berapa jenis denggi                What kind of dengue
4  Apakah punca demam denggi  What is the cause of dengue fever


In [7]:
# === Add tilde to each new line (Answer Only for Multi-Line) ===
def tilde_each_line(text):
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    if len(lines) <= 1:
        return lines[0] if lines else ''
    return '\n'.join(['~ ' + line for line in lines])

df['Answer (Malay)'] = df['Answer (Malay)'].apply(tilde_each_line)
df['Answer (English)'] = df['Answer (English)'].apply(tilde_each_line)
print("\n=== Preview: Tilde-Formatted Answers ===")
for i, row in enumerate(df[['Answer (Malay)', 'Answer (English)']].head(20).itertuples(index=False), 1):
    print(f"--- Answer {i} (Malay) ---\n{row[0]}\n")
    print(f"--- Answer {i} (English) ---\n{row[1]}\n")


=== Preview: Tilde-Formatted Answers ===
--- Answer 1 (Malay) ---
Ia adalah penyakit berpotensi maut yang tersebar oleh nyamuk Aedes Aegypti yang dijangkiti.

--- Answer 1 (English) ---
It is a potentially deadly disease spread by infected aedes aedes mosquitoes.

--- Answer 2 (Malay) ---
Demam denggi adalah penyakit yang disebarkan oleh nyamuk, dan ia memberi kesan kepada sekitar 400 juta orang di seluruh dunia setiap tahun.

--- Answer 2 (English) ---
Dengue fever is a disease spread by mosquitoes, and it affects about 400 million people worldwide every year.

--- Answer 3 (Malay) ---
Demam denggi adalah sejenis penyakit jangkitan virus yang merebak melalui gigitan nyamuk.

--- Answer 3 (English) ---
Dengue fever is a viral infection that spreads through mosquito bites.

--- Answer 4 (Malay) ---
Terdapat empat jenis virus yang boleh menyebabkan demam denggi iaitu DEN 1, DEN 2, DEN 3 dan DEN 4.

--- Answer 4 (English) ---
There are four types of viruses that can cause dengue fever: d

In [8]:
# === Lowercasing ===
df['Question (Malay)'] = df['Question (Malay)'].str.lower()
df['Question (English)'] = df['Question (English)'].str.lower()
print("\n=== Preview: Lowercased Text ===")
print(df[['Question (Malay)', 'Question (English)']].head())


=== Preview: Lowercased Text ===
            Question (Malay)                 Question (English)
0              apakah denggi                     what is dengue
1             apa itu denggi                     what is dengue
2        apakah demam denggi               what is dengue fever
3        berapa jenis denggi                what kind of dengue
4  apakah punca demam denggi  what is the cause of dengue fever


In [9]:
# === Tokenization ===
# Malay
tokenizer = malaya.tokenizer.Tokenizer()
df['Q_Tokens'] = df['Question (Malay)'].apply(tokenizer.tokenize)
df['A_Tokens'] = df['Answer (Malay)'].apply(tokenizer.tokenize)
print("\n=== Preview: Tokenized Questions & Answers ===")
print(df[['Q_Tokens', 'A_Tokens']].head())

# English
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

df['Q_Tokens_Eng'] = df['Question (English)'].apply(word_tokenize)
df['A_Tokens_Eng'] = df['Answer (English)'].apply(word_tokenize)
print(df[['Q_Tokens_Eng', 'A_Tokens_Eng']].head())


=== Preview: Tokenized Questions & Answers ===
                         Q_Tokens  \
0                [apakah, denggi]   
1              [apa, itu, denggi]   
2         [apakah, demam, denggi]   
3         [berapa, jenis, denggi]   
4  [apakah, punca, demam, denggi]   

                                            A_Tokens  
0  [Ia, adalah, penyakit, berpotensi, maut, yang,...  
1  [Demam, denggi, adalah, penyakit, yang, diseba...  
2  [Demam, denggi, adalah, sejenis, penyakit, jan...  
3  [Terdapat, empat, jenis, virus, yang, boleh, m...  
4  [Demam, denggi, adalah, jangkitan, virus, yang...  


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


                                Q_Tokens_Eng  \
0                         [what, is, dengue]   
1                         [what, is, dengue]   
2                  [what, is, dengue, fever]   
3                   [what, kind, of, dengue]   
4  [what, is, the, cause, of, dengue, fever]   

                                        A_Tokens_Eng  
0  [It, is, a, potentially, deadly, disease, spre...  
1  [Dengue, fever, is, a, disease, spread, by, mo...  
2  [Dengue, fever, is, a, viral, infection, that,...  
3  [There, are, four, types, of, viruses, that, c...  
4  [Dengue, fever, is, a, viral, infection, cause...  


In [10]:
# === Stopword Removal ===
# Malay
import malaya
from malaya.text.function import get_stopwords
malay_stopwords = set(get_stopwords())

df['Q_Tokens_Nostop'] = df['Q_Tokens'].apply(lambda toks: [t for t in toks if t not in malay_stopwords])
print("\n=== Preview: After Stopword Removal ===")
print(df[['Q_Tokens_Nostop']].head())

# English
from nltk.corpus import stopwords
nltk.download('stopwords')

english_stopwords = set(stopwords.words('english'))
df['Q_Tokens_Eng_Nostop'] = df['Q_Tokens_Eng'].apply(lambda toks: [t for t in toks if t.lower() not in english_stopwords])
print(df[['Q_Tokens_Eng_Nostop']].head())


=== Preview: After Stopword Removal ===
          Q_Tokens_Nostop
0                [denggi]
1                [denggi]
2         [demam, denggi]
3         [jenis, denggi]
4  [punca, demam, denggi]
      Q_Tokens_Eng_Nostop
0                [dengue]
1                [dengue]
2         [dengue, fever]
3          [kind, dengue]
4  [cause, dengue, fever]


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
# === Stemming/Lemmatization ===
# Malay
stemmer = malaya.stem.sastrawi()

df['Q_Tokens_Stem'] = df['Q_Tokens_Nostop'].apply(lambda toks: [stemmer.stem(t) for t in toks])
print("\n=== Preview: After Stemming ===")
print(df[['Q_Tokens_Stem']].head())

# English
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')
wnl = WordNetLemmatizer()

df['Q_Tokens_Eng_Lemma'] = df['Q_Tokens_Eng_Nostop'].apply(lambda toks: [wnl.lemmatize(t) for t in toks])
print(df[['Q_Tokens_Eng_Lemma']].head())

C:\Users\user\anaconda3\envs\python310\lib\site-packages\malaya\model\stem.py:28: FutureWarning: Possible nested set at position 3
  or re.findall(_expressions['ic'], word.lower())
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



=== Preview: After Stemming ===
            Q_Tokens_Stem
0                [denggi]
1                [denggi]
2         [demam, denggi]
3         [jenis, denggi]
4  [punca, demam, denggi]
       Q_Tokens_Eng_Lemma
0                [dengue]
1                [dengue]
2         [dengue, fever]
3          [kind, dengue]
4  [cause, dengue, fever]


In [12]:
# === Save Cleaned Data ===
df.to_csv('../Data/Dengue_QA_multilingual_cleaned.csv', index=False)
print('\nData preprocessing complete! Cleaned file saved as Dengue_QA_multilingual_cleaned.csv')


Data preprocessing complete! Cleaned file saved as Dengue_QA_multilingual_cleaned.csv
